In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
from datetime import datetime
import re

options = Options()
#options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

def extraer_info_producto_movistar(product, fecha_consulta):
    try:
        # Nombre: img alt tiene el nombre completo con capacidad y color
        nombre = ""
        img_tag = product.find("img", class_="grid__img")
        if img_tag:
            nombre = img_tag.get("alt", "").strip()
        if not nombre:
            brand_tag = product.find("span", class_="product-details--brand")
            model_tag = product.find("h3", class_="product-details--name")
            b = brand_tag.get_text(strip=True) if brand_tag else ""
            m = model_tag.get_text(strip=True) if model_tag else ""
            nombre = f"{b} {m}".strip()

        # URL
        url_producto = ""
        href = product.get("href", "")
        if href.startswith("http"):
            url_producto = href
        elif href.startswith("/"):
            url_producto = "https://tienda.movistar.com.mx" + href

        # SKU desde onclick: handleProductClick('id','TMLMX...','nombre',...)
        sku = ""
        onclick_text = product.get("onclick", "")
        if onclick_text:
            m = re.search(r"handleProductClick\('[^']*',\s*'([^']*)'", onclick_text)
            if m:
                sku = m.group(1)
        if not sku:
            sku = product.get("data-dl-product-view-detail", "")

        # Marca desde span.product-details--brand
        marca = ""
        brand_tag = product.find("span", class_="product-details--brand")
        if brand_tag:
            brand_raw = brand_tag.get_text(strip=True).upper()
            marca_map = {
                "APPLE": "APPLE", "IPHONE": "APPLE",
                "SAMSUNG": "SAMSUNG", "GALAXY": "SAMSUNG",
                "XIAOMI": "XIAOMI", "REDMI": "XIAOMI", "POCO": "XIAOMI",
                "MOTOROLA": "MOTOROLA", "MOTO": "MOTOROLA",
                "HONOR": "HONOR", "OPPO": "OPPO", "HUAWEI": "HUAWEI",
                "NOKIA": "NOKIA", "TCL": "TCL", "GOOGLE": "GOOGLE",
                "VIVO": "VIVO", "REALME": "REALME", "ONEPLUS": "ONEPLUS",
            }
            marca = marca_map.get(brand_raw, brand_raw)
        if not marca and nombre:
            nombre_lower = nombre.lower()
            for key, val in {
                'iphone': 'APPLE', 'apple': 'APPLE',
                'samsung': 'SAMSUNG', 'galaxy': 'SAMSUNG',
                'xiaomi': 'XIAOMI', 'redmi': 'XIAOMI', 'poco': 'XIAOMI',
                'motorola': 'MOTOROLA', 'moto': 'MOTOROLA',
                'honor': 'HONOR', 'oppo': 'OPPO', 'huawei': 'HUAWEI',
                'nokia': 'NOKIA', 'tcl': 'TCL', 'google': 'GOOGLE',
                'vivo': 'VIVO', 'realme': 'REALME',
            }.items():
                if key in nombre_lower:
                    marca = val
                    break

        # Color
        color = ""
        if nombre:
            nombre_lower = nombre.lower()
            colores = [
                'negro', 'blanco', 'azul', 'rojo', 'verde', 'amarillo', 'rosa',
                'morado', 'gris', 'dorado', 'plateado', 'violeta', 'naranja',
                'celeste', 'turquesa', 'titanio', 'cyan', 'café', 'plata',
                'lila', 'beige', 'coral', 'lavanda', 'crema', 'esmeralda',
                'arándano', 'olivo', 'black', 'white', 'blue', 'red',
            ]
            for c in colores:
                if c in nombre_lower:
                    color = c.capitalize()
                    break

        # Capacidad
        capacidad = ""
        if nombre:
            cap_m = re.search(r'(\d+(?:\.\d+)?)\s*(GB|TB)', nombre, re.IGNORECASE)
            if cap_m:
                capacidad = f"{cap_m.group(1)} {cap_m.group(2).upper()}"

        def limpiar_precio(texto):
            if texto:
                return re.sub(r'[^\d,.]', '', texto.replace('$', '').replace('MXN', '')).strip()
            return ""

        # Precios
        precio_original = ""
        precio_descuento = "No discount"
        porcentaje_str = "No discount"

        precio_tachado_tag = product.find("span", class_="product-price--amount-crossed")
        precio_actual_tag = product.find("span", class_="product-price--amount")
        precio_actual = limpiar_precio(precio_actual_tag.get_text(strip=True)) if precio_actual_tag else ""

        if precio_tachado_tag:
            # Hay descuento: tachado = precio original, amount = precio final
            precio_original = limpiar_precio(precio_tachado_tag.get_text(strip=True))
            precio_descuento = precio_actual if precio_actual else "No discount"
            # Porcentaje desde el HTML
            pct_tag = product.find("span", class_="product-price--discount-percent")
            if pct_tag:
                pct_m = re.search(r'(\d+)%', pct_tag.get_text(strip=True))
                if pct_m:
                    porcentaje_str = f"{pct_m.group(1)}%"
            elif precio_original and precio_descuento != "No discount":
                try:
                    p_ori = float(precio_original.replace(',', ''))
                    p_desc = float(precio_descuento.replace(',', ''))
                    if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
                        porcentaje_str = f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
                except Exception:
                    pass
        elif precio_actual:
            # Sin descuento
            precio_original = precio_actual
            precio_descuento = "No discount"
        else:
            precio_original = "No inventory"
            precio_descuento = "No discount"

        return {
            "Marca": marca,
            "Nombre": nombre,
            "Color": color,
            "Capacidad": capacidad,
            "SKU": sku,
            "URL": url_producto,
            "FechaConsulta": fecha_consulta,
            "Precio_original": precio_original,
            "Precio_descuento": precio_descuento,
            "Porcentaje_descuento": porcentaje_str
        }

    except Exception as e:
        print(f"Error al extraer producto: {e}")
        return None

def obtener_total_paginas(soup):
    toolbar = soup.find("p", class_="toolbar-amount")
    if toolbar:
        texto = toolbar.get_text(strip=True)
        match = re.search(r'de\s+(\d+)', texto)
        if match:
            total_productos = int(match.group(1))
            total_paginas = (total_productos + 11) // 12
            print(f"Total de productos detectados: {total_productos}")
            print(f"Total de páginas calculadas: {total_paginas}")
            return total_paginas
    page_links = soup.select("ul.items.pages-items li.item a")
    if page_links:
        numeros = [int(l.get_text(strip=True)) for l in page_links if l.get_text(strip=True).isdigit()]
        if numeros:
            max_page = max(numeros)
            if max_page > 1:
                print(f"Total de páginas detectadas por paginación: {max_page}")
                return max_page
    all_text = soup.get_text()
    match = re.search(r'(\d+)-(\d+)\s+de\s+(\d+)', all_text)
    if match:
        total_productos = int(match.group(3))
        total_paginas = (total_productos + 11) // 12
        print(f"Total productos (método 3): {total_productos}, páginas: {total_paginas}")
        return total_paginas
    return None

def scrape_movistar_smartphones():
    base_url = "https://tienda.movistar.com.mx/telefonos.html"

    print("=" * 60)
    print("INICIANDO SCRAPING DE MOVISTAR")
    print("=" * 60)

    driver = webdriver.Chrome(options=options)
    data = []
    fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    productos_procesados = set()
    max_paginas = 30

    try:
        print(f"Cargando página inicial: {base_url}")
        driver.get(base_url)
        time.sleep(5)
        for _ in range(3):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1)

        soup_inicial = BeautifulSoup(driver.page_source, "html.parser")
        total_paginas = obtener_total_paginas(soup_inicial)

        if total_paginas:
            max_paginas = min(total_paginas, 30)
            print(f"Se procesarán {max_paginas} páginas")
        else:
            max_paginas = 20
            print(f"No se detectó total de páginas, se intentarán hasta {max_paginas}")

        paginas_vacias_consecutivas = 0

        for page_number in range(1, max_paginas + 1):
            print(f"\n{'='*50}")
            print(f"PÁGINA {page_number} de {max_paginas}")
            print(f"{'='*50}")

            if page_number == 1:
                page_url = base_url
            else:
                page_url = f"{base_url}?p={page_number}"

            if page_number > 1:
                print(f"Navegando a: {page_url}")
                driver.get(page_url)
                WebDriverWait(driver, 15).until(
                    lambda d: d.execute_script("return document.readyState") == "complete"
                )
                time.sleep(3)
                for _ in range(3):
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(1)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            products = soup.find_all("a", {"data-dl-product-view-detail": True})
            print(f"Productos encontrados en página {page_number}: {len(products)}")

            if len(products) == 0:
                paginas_vacias_consecutivas += 1
                print(f"Página sin productos (consecutivas: {paginas_vacias_consecutivas})")
                if paginas_vacias_consecutivas >= 2:
                    print("2 páginas vacías consecutivas. Finalizando...")
                    break
                continue
            else:
                paginas_vacias_consecutivas = 0

            # Debug primera página
            if page_number == 1:
                for i, product in enumerate(products[:2]):
                    print(f"\n--- Debug Producto {i+1} ---")
                    img_tag = product.find("img", class_="grid__img")
                    if img_tag:
                        print(f"Nombre (alt): {img_tag.get('alt', '')}")
                    onclick = product.get("onclick", "")
                    m = re.search(r"handleProductClick\('[^']*',\s*'([^']*)'", onclick)
                    if m:
                        print(f"SKU: {m.group(1)}")
                    precio_tachado = product.find("span", class_="product-price--amount-crossed")
                    precio_actual = product.find("span", class_="product-price--amount")
                    if precio_tachado:
                        print(f"Precio original: {precio_tachado.get_text(strip=True)}")
                    if precio_actual:
                        print(f"Precio descuento: {precio_actual.get_text(strip=True)}")

            productos_pagina_actual = 0
            for product in products:
                onclick = product.get("onclick", "")
                m = re.search(r"handleProductClick\('[^']*',\s*'([^']*)'", onclick)
                sku_temp = m.group(1) if m else product.get("data-dl-product-view-detail", "")

                if sku_temp and sku_temp in productos_procesados:
                    continue

                info = extraer_info_producto_movistar(product, fecha_consulta)
                if info and info["Nombre"] and len(info["Nombre"]) > 5 and info["Marca"]:
                    data.append(info)
                    if sku_temp:
                        productos_procesados.add(sku_temp)
                    productos_pagina_actual += 1
                    desc = f" | Dcto: ${info['Precio_descuento']} ({info['Porcentaje_descuento']})" if info['Porcentaje_descuento'] != 'No discount' else ""
                    print(f"  ✅ {info['Nombre'][:50]} | SKU: {info['SKU']} | ${info['Precio_original']}{desc}")

            print(f"\nNuevos en página {page_number}: {productos_pagina_actual}")
            print(f"Total acumulado: {len(data)}")

        print(f"\n{'='*60}")
        print(f"SCRAPING COMPLETADO")
        print(f"Páginas procesadas: {page_number}")
        print(f"Productos únicos extraídos: {len(data)}")
        print(f"{'='*60}")

    except Exception as e:
        print(f"Error durante el scraping: {e}")
        import traceback
        traceback.print_exc()
    finally:
        driver.quit()

    return data

# Ejecutar
resultados_movistar = scrape_movistar_smartphones()

if resultados_movistar:
    df_movistar = pd.DataFrame(resultados_movistar)
    print(f"\nProductos antes de limpiar: {len(df_movistar)}")
    df_movistar = df_movistar[df_movistar['Nombre'].str.len() > 5]
    df_movistar = df_movistar[df_movistar['Marca'].str.len() > 0]
    df_movistar = df_movistar.drop_duplicates(subset=['SKU'], keep='first')
    df_movistar = df_movistar.sort_values(['Marca', 'Nombre']).reset_index(drop=True)
    print(f"Productos después de limpiar: {len(df_movistar)}")

    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"movistar_smartphones_completo_{fecha_hoy}.csv"
    df_movistar.to_csv(nombre_archivo, index=False, encoding="utf-8")
    print(f"\nDatos exportados a {nombre_archivo}")
    print(f"Total de productos únicos: {len(df_movistar)}")

    print("\n=== ESTADÍSTICAS FINALES ===")
    print(f"Total de productos: {len(df_movistar)}")
    print(f"\nProductos por marca:")
    print(df_movistar['Marca'].value_counts())
    print(f"\nProductos con descuento: {len(df_movistar[df_movistar['Porcentaje_descuento'] != 'No discount'])}")
    print("\n=== PRIMEROS 10 PRODUCTOS ===")
    print(df_movistar.head(10))
else:
    print("No se encontraron productos válidos.")


INICIANDO SCRAPING DE MOVISTAR
Cargando página inicial: https://tienda.movistar.com.mx/telefonos.html
Total productos (método 3): 115, páginas: 10
Se procesarán 10 páginas

PÁGINA 1 de 10
Productos encontrados en página 1: 12

--- Debug Producto 1 ---
Nombre (alt): Oppo A6T 256 GB Blanco
SKU: TMLMXOPOPA6TBL28
Precio descuento: $6,999

--- Debug Producto 2 ---
Nombre (alt): Oppo A6T 256 GB Oliva
SKU: TMLMXOPOPA6TOL28
Precio descuento: $6,999
  ✅ Oppo A6T 256 GB Blanco | SKU: TMLMXOPOPA6TBL28 | $6,999
  ✅ Oppo A6T 256 GB Oliva | SKU: TMLMXOPOPA6TOL28 | $6,999
  ✅ Oppo A6K 256 GB Azul | SKU: TMLMXOPOPA6KAZ00 | $7,499
  ✅ Oppo A6K 256 GB Morado | SKU: TMLMXOPOPA6KMO00 | $7,499
  ✅ OPPO Reno16 F 5G 256 GB Morado | SKU: TMLMXOPRE16FMO56 | $15,999
  ✅ Honor Magic8 Lite 512 GB 5G Negro con Licuadora po | SKU: TMLMXHNM8LBMNE12 | $9,999
  ✅ OPPO Reno16 5G 512 GB Blanco | SKU: TMLMXOPR165GBL56 | $18,999
  ✅ OPPO Reno16 F 5G 256 GB Blanco | SKU: TMLMXOPRE16FBL56 | $15,999
  ✅ Honor X7d 256 GB Dora